# module-extra-repr — worked example 1: extra_repr for a Conv-style block

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-extra-repr`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Overriding `extra_repr` on an `nn.Module` lets you control the text that appears inside the parentheses when the module is printed. PyTorch calls it automatically during `__repr__`, so you return only the descriptive string, not the whole layout. The classic gotcha is dumping a raw tensor into the string instead of a small scalar summary.

## Worked solution

We build a tiny `MyConvBlock` whose only interesting behavior is its repr. In `__init__` we call `super().__init__()` first so the Module machinery is ready before we assign anything. We store the config integers `in_channels`, `out_channels`, and `kernel_size` as plain attributes, and we register a `weight` Parameter shaped `(out_channels, in_channels, kernel_size, kernel_size)` (zeros, since initialization is not the focus). The key step is `extra_repr`, which returns an f-string listing the three config integers. We deliberately surface only the integers and never the weight tensor itself, because `print(mod)` would then dump the full tensor. Finally we instantiate the block and print it: PyTorch wraps our `extra_repr` string inside `MyConvBlock(...)` for us.

In [ ]:
import torch as t

class MyConvBlock(t.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.weight = t.nn.Parameter(
            t.zeros(out_channels, in_channels, kernel_size, kernel_size)
        )

    def extra_repr(self):
        return (f'in_channels={self.in_channels}, '
                f'out_channels={self.out_channels}, '
                f'kernel_size={self.kernel_size}')

    def forward(self, x):
        return x

block = MyConvBlock(3, 16, 5)
print(repr(block))